# Week 2 — Financial Data and Returns: Executable Lesson

[Open this notebook in Google Colab](https://colab.research.google.com/github/KennethWYLee/FinTech/blob/main/course/resources/notebooks/05_Week2_Financial_Data_and_Returns.ipynb)

This notebook develops the twenty Week 2 issues through explanations, executable examples, output interpretation, discussion questions, and evidence checks. It accompanies the complete lesson description in [Week 2 main](../../02/week2_main.md).

The sequence stops before technical indicators, trading signals, prediction models, and backtesting. All required calculations use fixed artificial data. Issue 2 contains an optional live yfinance query so that students can examine a real retrieval process without making the rest of the lesson depend on internet access.

Run the notebook from top to bottom. Keep every prediction, output, revision, and unresolved question. A successful execution confirms only that the documented calculations ran in the stated environment; it does not validate a real provider or establish investment value.

## Working rules

- Use decimal returns in calculations. Convert to percentages only for communication.
- Treat every asset, market, currency, and value labeled artificial as an invented teaching example.
- Do not silently repair a value. Preserve the original, explain the problem, and state the rule used to create any revised version.
- A provider label is not a definition. Real-data work requires the provider documentation and retrieval record.
- Do not commit downloaded Yahoo Finance rows or notebook output containing those rows to a public repository unless their permitted use and redistribution have been verified.

In [1]:
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
    YFINANCE_VERSION = version("yfinance")
except (ImportError, PackageNotFoundError):
    yf = None
    YFINANCE_AVAILABLE = False
    YFINANCE_VERSION = "not installed"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

environment = pd.Series(
    {
        "Python interface": "Jupyter kernel",
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "yfinance": YFINANCE_VERSION,
        "yfinance_available": YFINANCE_AVAILABLE,
    },
    name="environment",
)
display(environment)

Python interface      Jupyter kernel
numpy                          2.2.6
pandas                         2.2.3
yfinance                       1.2.0
yfinance_available              True
Name: environment, dtype: object

## Issue 1 — What does one financial-data row represent?


A row is an observation recorded under a convention. It is not automatically a trade, a complete market session, or a value that was publicly available at the timestamp shown. A number becomes interpretable only when the asset, market or venue, field, unit, currency, timestamp meaning, and frequency are known.

The example contrasts an ambiguous two-field record with a documented observation. The documented row is more useful, but its additional fields still do not prove that the value is correct. Metadata improve interpretation; source and validity checks remain necessary.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [2]:
record_a = {
    "Date": "2026-01-05",
    "Value": 100.0,
}

record_b = {
    "Asset": "Asset_A",
    "Market": "Artificial market",
    "Timestamp": "2026-01-05 16:00",
    "Timestamp_meaning": "Artificial session close",
    "Field": "Closing price",
    "Value": 100.0,
    "Unit": "Artificial currency units per share",
    "Currency": "ACU",
    "Frequency": "One observation per artificial session",
}

required_meaning = {
    "Asset",
    "Market",
    "Timestamp",
    "Timestamp_meaning",
    "Field",
    "Unit",
    "Currency",
    "Frequency",
}

comparison = pd.DataFrame(
    {
        "Record A": pd.Series(record_a),
        "Record B": pd.Series(record_b),
    }
)
display(comparison)

missing_a = sorted(required_meaning.difference(record_a))
missing_b = sorted(required_meaning.difference(record_b))
print("Missing meaning in Record A:", missing_a)
print("Missing meaning in Record B:", missing_b)

assert len(missing_a) == 8
assert missing_b == []

,Record A,Record B
Asset,NaN,Asset_A
Currency,NaN,ACU
Date,2026-01-05,NaN
Field,NaN,Closing price
Frequency,NaN,One observation per artificial session
Market,NaN,Artificial market
Timestamp,NaN,2026-01-05 16:00
Timestamp_meaning,NaN,Artificial session close
Unit,NaN,Artificial currency units per share
Value,100.0,100.0


Missing meaning in Record A: ['Asset', 'Currency', 'Field', 'Frequency', 'Market', 'Timestamp', 'Timestamp_meaning', 'Unit']
Missing meaning in Record B: []


### Interpret the output


Record A permits several incompatible interpretations of the number 100. Record B narrows the interpretation to one artificial closing-price observation. Notice that Record B still does not state a provider, retrieval date, or method used to verify the value.


### Discussion questions

1. Name three different financial quantities that the value 100 in Record A could represent.
2. Why does a date label alone fail to identify when information became available?
3. Which metadata describe economic meaning, and which metadata would instead support provenance?
4. Would Record B become a transaction record merely by adding a volume field? Explain.

### Evidence to preserve

Save both records, the two missing-field lists, your original prediction, and a revised one-sentence definition of a financial-data row.

### First diagnostic action

When a row remains ambiguous, list every missing field before calculating a return.

## Issue 2 — Which source and retrieval record support the data?


Reproducibility requires more than a URL. A retrieval record should identify the library or access method, provider, symbol, date range, interval, adjustment and corporate-action settings, package version, retrieval time, and any redistribution limitation that affects the submitted work.

yfinance is an open-source interface to Yahoo Finance data; it is not an official Yahoo product. Review the [yfinance project notice](https://github.com/ranaroussi/yfinance/blob/main/README.md) and the current [download-function documentation](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html) before using the query. If yfinance is unavailable in Colab, install it in a separate cell with %pip install yfinance and then rerun the environment cell.

The live query below is disabled by default. Students may enable it in a connected environment, inspect only a small result, and preserve the query record rather than publishing the downloaded rows. The fixed artificial examples in later issues do not depend on this query.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [3]:
query_record = pd.Series(
    {
        "access_library": "yfinance",
        "library_version": YFINANCE_VERSION,
        "provider_named_by_library": "Yahoo Finance",
        "symbol": "MSFT",
        "start": "2024-01-02",
        "end": "2024-02-01",
        "interval": "1d",
        "auto_adjust": False,
        "actions": True,
        "retrieval_time_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "raw_rows_permitted_in_public_submission": "not established by this notebook",
    },
    name="query_record",
)
display(query_record)

RUN_LIVE_DOWNLOAD = False
live_download = None

if not RUN_LIVE_DOWNLOAD:
    print("Live download skipped. Set RUN_LIVE_DOWNLOAD = True in a connected class environment.")
elif not YFINANCE_AVAILABLE:
    print("Live download skipped because yfinance is not installed.")
else:
    try:
        live_download = yf.download(
            tickers=query_record["symbol"],
            start=query_record["start"],
            end=query_record["end"],
            interval=query_record["interval"],
            auto_adjust=query_record["auto_adjust"],
            actions=query_record["actions"],
            progress=False,
            threads=False,
            multi_level_index=False,
        )
        print("Downloaded shape:", live_download.shape)
        display(live_download.head(3))
        print("Columns:", list(live_download.columns))
    except Exception as exc:
        print("Live download failed; preserve this message and continue with artificial data.")
        print(type(exc).__name__, str(exc)[:300])

assert query_record["auto_adjust"] is False
assert query_record["actions"] is True

access_library                                                     yfinance
library_version                                                       1.2.0
provider_named_by_library                                     Yahoo Finance
symbol                                                                 MSFT
start                                                            2024-01-02
end                                                              2024-02-01
interval                                                                 1d
auto_adjust                                                           False
actions                                                                True
retrieval_time_utc                         2026-08-28T07:53:32.654330+00:00
raw_rows_permitted_in_public_submission    not established by this notebook
Name: query_record, dtype: object

Live download skipped. Set RUN_LIVE_DOWNLOAD = True in a connected class environment.


### Interpret the output


The query record is available even when the network call is skipped or fails. This separation is deliberate: a reproducible specification records what was requested, while the retrieval result records what the service returned at that time. Repeating the same query later may produce a different row set because providers can revise, correct, or reformat data.


### Discussion questions

1. What is the difference between yfinance as an access library and Yahoo Finance as the named data provider?
2. Why must auto_adjust and actions be recorded rather than left at undocumented defaults?
3. Which parts of the result might change if the same query is repeated next month?
4. What evidence should be submitted if the live request fails during class?

### Evidence to preserve

Save the query record, package version, live-query status, column names when available, and the exact failure message when unavailable. Do not save raw downloaded rows to the public course repository.

### First diagnostic action

When a retrieval is empty or fails, inspect the symbol, dates, interval, package version, network status, and complete error message before changing the financial question.

## Issue 3 — Have you identified the asset rather than only a short symbol?


A short symbol is convenient but incomplete. Symbols may be reused across venues, changed over time, or used for different security types. An asset record should preserve the provider identifier, descriptive name, security type, market or venue, trading currency, and the date range for which the mapping is intended.

The artificial asset master below contains one complete record, one incomplete record, and two rows that reuse the same short symbol on different venues. The duplicate symbol is not automatically an error; it is evidence that the symbol alone is not a unique key.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [4]:
asset_master = pd.DataFrame(
    [
        ["AAA", "Artificial Alpha", "Common stock", "Venue_X", "ACU", "2026-01-01", None],
        ["BBB", "Artificial Beta", None, "Venue_X", "ACU", "2026-01-01", None],
        ["XYZ", "Artificial X One", "Fund", "Venue_X", "ACU", "2026-01-01", None],
        ["XYZ", "Artificial X Two", "Common stock", "Venue_Y", "BCU", "2026-01-01", None],
    ],
    columns=[
        "Symbol",
        "Asset_name",
        "Security_type",
        "Venue",
        "Currency",
        "Mapping_start",
        "Mapping_end",
    ],
)

asset_master["Composite_key"] = (
    asset_master["Symbol"] + "|" + asset_master["Venue"] + "|" + asset_master["Currency"]
)

display(asset_master)
print("Duplicated short symbols:", asset_master["Symbol"].duplicated(keep=False).sum())
print("Duplicated composite keys:", asset_master["Composite_key"].duplicated().sum())
print("Missing security types:", asset_master["Security_type"].isna().sum())

assert asset_master["Symbol"].duplicated(keep=False).sum() == 2
assert asset_master["Composite_key"].is_unique
assert asset_master["Security_type"].isna().sum() == 1

,Symbol,Asset_name,Security_type,Venue,Currency,Mapping_start,Mapping_end,Composite_key
0,AAA,Artificial Alpha,Common stock,Venue_X,ACU,2026-01-01,None,AAA|Venue_X|ACU
1,BBB,Artificial Beta,None,Venue_X,ACU,2026-01-01,None,BBB|Venue_X|ACU
2,XYZ,Artificial X One,Fund,Venue_X,ACU,2026-01-01,None,XYZ|Venue_X|ACU
3,XYZ,Artificial X Two,Common stock,Venue_Y,BCU,2026-01-01,None,XYZ|Venue_Y|BCU


Duplicated short symbols: 2
Duplicated composite keys: 0
Missing security types: 1


### Interpret the output


The two XYZ rows show why a join based only on Symbol could attach prices to the wrong instrument. The incomplete BBB row should remain pending until security type is verified; filling it by analogy would create unsupported metadata.


### Discussion questions

1. Which columns are required to distinguish the two XYZ records?
2. How could a symbol change create a false price jump in a long historical file?
3. Why should an index level, fund price, futures settlement, and stock close not share one undocumented price definition?
4. When would a composite key still be insufficient to establish historical identity?

### Evidence to preserve

Save the asset master, duplicate-symbol count, composite-key check, missing-field count, and a sentence stating which field must remain pending.

### First diagnostic action

When two sources use the same symbol, compare provider identifiers, venue, currency, security type, and effective dates before joining.

## Issue 4 — What do Open, High, Low, Close, and Volume mean?


OHLCV fields summarize observations under a provider and session convention. Open and Close are the first and final included prices; High and Low are the largest and smallest included prices; Volume is a reported traded quantity under a stated unit. The words do not reveal whether extended sessions are included or whether volume counts shares, contracts, or another quantity.

Arithmetic range checks can detect contradictions, but they cannot verify the session definition. The example includes one intentionally invalid row whose High is below its Close.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [5]:
lesson_ohlcv = pd.DataFrame(
    {
        "Date": pd.to_datetime(["2026-01-05", "2026-01-06", "2026-01-07"]),
        "Open": [100.0, 102.0, 101.0],
        "High": [103.0, 101.0, 102.0],
        "Low": [99.5, 100.5, 100.0],
        "Close": [102.0, 102.5, 101.5],
        "Volume": [1_000_000, 1_100_000, 950_000],
    }
).set_index("Date")

ohlcv_audit = pd.DataFrame(index=lesson_ohlcv.index)
ohlcv_audit["low_not_above_open"] = lesson_ohlcv["Low"] <= lesson_ohlcv["Open"]
ohlcv_audit["low_not_above_close"] = lesson_ohlcv["Low"] <= lesson_ohlcv["Close"]
ohlcv_audit["high_not_below_open"] = lesson_ohlcv["High"] >= lesson_ohlcv["Open"]
ohlcv_audit["high_not_below_close"] = lesson_ohlcv["High"] >= lesson_ohlcv["Close"]
ohlcv_audit["volume_nonnegative"] = lesson_ohlcv["Volume"] >= 0
ohlcv_audit["row_passes_arithmetic"] = ohlcv_audit.all(axis=1)

display(lesson_ohlcv)
display(ohlcv_audit)
print("Rows failing arithmetic checks:", (~ohlcv_audit["row_passes_arithmetic"]).sum())

assert (~ohlcv_audit["row_passes_arithmetic"]).sum() == 1

,Open,High,Low,Close,Volume
Date,,,,,
2026-01-05,100.0,103.0,99.5,102.0,1000000
2026-01-06,102.0,101.0,100.5,102.5,1100000
2026-01-07,101.0,102.0,100.0,101.5,950000


,low_not_above_open,low_not_above_close,high_not_below_open,high_not_below_close,volume_nonnegative,row_passes_arithmetic
Date,,,,,,
2026-01-05,True,True,True,True,True,True
2026-01-06,True,True,False,False,True,False
2026-01-07,True,True,True,True,True,True


Rows failing arithmetic checks: 1


### Interpret the output


The January 6 row fails because the reported Close exceeds the reported High. The correct response is not to raise High or lower Close without evidence. Display the source row, check the provider definition, and determine whether fields from different sessions or instruments were combined.


### Discussion questions

1. Which OHLC relations are arithmetic necessities under a common session definition?
2. What session information cannot be learned from the arithmetic checks?
3. Why is a nonnegative volume check insufficient to establish the meaning of Volume?
4. What evidence would justify correcting the January 6 row?

### Evidence to preserve

Save the input table, every Boolean range check, the failed row, your prediction, and one provider question that arithmetic cannot answer.

### First diagnostic action

When a range relation fails, display the entire original row and source definition before proposing any repair.

## Issue 5 — Which price field answers the return question?


Raw close, adjusted close, and a total-return index can answer different questions. A raw close may represent the quoted closing level. An adjusted series may apply provider-defined transformations for corporate actions. A total-return index may incorporate distributions under a reinvestment rule. Labels are not universal, and two providers may construct similarly named fields differently.

The artificial values below deliberately produce different returns. They illustrate why field choice changes the calculation; they do not reproduce a real provider adjustment formula.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [6]:
field_demo = pd.DataFrame(
    {
        "Raw_Close": [100.0, 50.0, 51.0],
        "Provider_Adjusted_Close": [50.0, 50.0, 51.0],
        "Artificial_Total_Return_Index": [100.0, 100.5, 103.0],
    },
    index=pd.to_datetime(["2026-02-02", "2026-02-03", "2026-02-04"]),
)

field_returns = field_demo.pct_change(fill_method=None)
display(field_demo)
display(field_returns.round(6))

print("Raw-close return on second date:", field_returns["Raw_Close"].iloc[1])
print("Adjusted-close return on second date:", field_returns["Provider_Adjusted_Close"].iloc[1])
print("Total-return-index return on second date:", field_returns["Artificial_Total_Return_Index"].iloc[1])

assert np.isclose(field_returns["Raw_Close"].iloc[1], -0.50)
assert np.isclose(field_returns["Provider_Adjusted_Close"].iloc[1], 0.0)

,Raw_Close,Provider_Adjusted_Close,Artificial_Total_Return_Index
2026-02-02,100.0,50.0,100.0
2026-02-03,50.0,50.0,100.5
2026-02-04,51.0,51.0,103.0


,Raw_Close,Provider_Adjusted_Close,Artificial_Total_Return_Index
2026-02-02,NaN,NaN,NaN
2026-02-03,-0.50,0.00,0.005000
2026-02-04,0.02,0.02,0.024876


Raw-close return on second date: -0.5
Adjusted-close return on second date: 0.0
Total-return-index return on second date: 0.004999999999999893


### Interpret the output


The three columns do not describe the same change in value. The example therefore cannot support the instruction to always use one named field. The analysis must first state whether it seeks quoted price change, a provider-adjusted change, or a distribution-inclusive holding return.


### Discussion questions

1. Which field would answer a narrowly defined raw price-change question?
2. What provider documentation is required before using an adjusted field?
3. How could a distribution be counted twice when mixing adjusted prices and a separate dividend column?
4. Why is choosing the field with the best historical performance invalid?

### Evidence to preserve

Save the three field definitions you would need, the return table, and a written field decision tied to one financial question.

### First diagnostic action

When returns differ across fields, stop and compare documented construction rules rather than selecting a column by name or outcome.

## Issue 6 — How does a stock split affect price and share count?


A stock split can change the quoted price per share and the number of shares held without creating the same proportional change in total position value. A raw per-share return across the event can therefore be economically misleading if share-count change is ignored.

This simplified artificial example assumes a two-for-one split, no other price movement, no fees, and exact adjustment at the boundary. Real data require the provider event record and effective-time convention.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [7]:
split_example = pd.Series(
    {
        "price_before": 100.0,
        "shares_before": 1.0,
        "split_ratio_new_per_old": 2.0,
        "price_after": 50.0,
        "shares_after": 2.0,
    }
)

raw_per_share_return = split_example["price_after"] / split_example["price_before"] - 1
value_before = split_example["price_before"] * split_example["shares_before"]
value_after = split_example["price_after"] * split_example["shares_after"]
position_value_return = value_after / value_before - 1

display(split_example)
print("Raw per-share return:", raw_per_share_return)
print("Position value before:", value_before)
print("Position value after:", value_after)
print("Position-value return:", position_value_return)

assert np.isclose(raw_per_share_return, -0.50)
assert np.isclose(position_value_return, 0.0)

price_before               100.0
shares_before                1.0
split_ratio_new_per_old      2.0
price_after                 50.0
shares_after                 2.0
dtype: float64

Raw per-share return: -0.5
Position value before: 100.0
Position value after: 100.0
Position-value return: 0.0


### Interpret the output


The negative 50% raw per-share return and zero position-value return answer different questions. The example does not imply that every exact halving is a split; an authoritative corporate-action record is still required.


### Discussion questions

1. Why does the raw price return fail to describe the unchanged position value?
2. Which event fields are needed to verify the split ratio and effective date?
3. How would an incorrect split adjustment affect a momentum feature later in the course?
4. What double-counting error could occur if both adjusted prices and a manual split correction are applied?

### Evidence to preserve

Save the five inputs, both return calculations, the value identity, and a list of provider evidence needed for a real event.

### First diagnostic action

When a large price change coincides with a share-count event, compare raw price, adjusted field, split record, and position value before labeling it a loss.

## Issue 7 — How can a cash dividend change the return calculation?


A price return uses only beginning and ending prices. A simplified holding-period return can add a cash distribution received during the interval. The distribution timing, eligibility, tax treatment, reinvestment assumption, and currency must be defined before a real return is calculated.

The artificial example assumes one share, a cash distribution of 3 units, and no reinvestment or tax. It separates price change from distribution-inclusive value change.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [8]:
dividend_example = pd.Series(
    {
        "price_before": 100.0,
        "price_after": 98.0,
        "cash_distribution": 3.0,
    }
)

price_return = dividend_example["price_after"] / dividend_example["price_before"] - 1
holding_period_return = (
    dividend_example["price_after"] + dividend_example["cash_distribution"]
) / dividend_example["price_before"] - 1

display(dividend_example)
print("Price return:", price_return)
print("Simplified holding-period return:", holding_period_return)

assert np.isclose(price_return, -0.02)
assert np.isclose(holding_period_return, 0.01)

price_before         100.0
price_after           98.0
cash_distribution      3.0
dtype: float64

Price return: -0.020000000000000018
Simplified holding-period return: 0.010000000000000009


### Interpret the output


The price fell by 2%, while price plus the assumed cash distribution increased value by 1%. The simplified calculation is not automatically a total-return series because it omits reinvestment timing, taxes, currency conversion, and provider conventions.


### Discussion questions

1. What eligibility condition must be satisfied for the investor to receive the cash distribution?
2. How would reinvestment timing change the required inputs?
3. When could an adjusted price plus a separate dividend create double counting?
4. Why should price return and holding-period return be labeled separately in a report?

### Evidence to preserve

Save the inputs, both returns, assumptions about distribution timing and reinvestment, and one limitation of the simplified formula.

### First diagnostic action

When a dividend-inclusive result looks unexpectedly high, inspect whether the price field already incorporates distributions.

## Issue 8 — Are units, currency, and percentage language consistent?


Many financial errors are unit errors rather than formula errors. A decimal return of 0.01, a percentage return of 1%, and 100 basis points describe the same change. A change from 4% to 5% is an increase of one percentage point, not one percent.

Currency and price units require the same discipline. A portfolio cannot combine values denominated in different currencies without documenting whether and how conversion occurs.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [9]:
unit_examples = pd.DataFrame(
    {
        "Input_label": ["decimal return", "percentage return", "basis points", "rate change"],
        "Input_value": [0.01, 1.0, 100.0, 5.0 - 4.0],
        "Converted_decimal": [0.01, 1.0 / 100, 100.0 / 10_000, np.nan],
        "Communication": ["1%", "1%", "1%", "1 percentage point"],
    }
)

display(unit_examples)
print("100 basis points in decimal units:", 100 / 10_000)
print("Rate change from 4% to 5%:", 5 - 4, "percentage point")
print("Relative increase from 4% to 5%:", 5.0 / 4.0 - 1.0)

assert np.isclose(unit_examples.loc[0, "Converted_decimal"], 0.01)
assert np.isclose(unit_examples.loc[1, "Converted_decimal"], 0.01)
assert np.isclose(unit_examples.loc[2, "Converted_decimal"], 0.01)
assert np.isclose(5.0 / 4.0 - 1.0, 0.25)

,Input_label,Input_value,Converted_decimal,Communication
0,decimal return,0.01,0.01,1%
1,percentage return,1.00,0.01,1%
2,basis points,100.00,0.01,1%
3,rate change,1.00,NaN,1 percentage point


100 basis points in decimal units: 0.01
Rate change from 4% to 5%: 1 percentage point
Relative increase from 4% to 5%: 0.25


### Interpret the output


The first three rows reconcile to the same decimal return. Moving from 4% to 5% is a change of one percentage point and a relative increase of 25%; those statements use different bases. The rate-change row should not be silently converted into an investment return. Units must appear beside both inputs and outputs.


### Discussion questions

1. What numerical mistake occurs when 1% is entered as 1 rather than 0.01?
2. Why is a rise from 4% to 5% not described as a one-percent increase?
3. Which exchange-rate convention would be needed before combining two currencies?
4. Where should units be recorded so they remain visible after columns are renamed?

### Evidence to preserve

Save the unit-conversion table, one corrected unit error, and a currency rule for any multi-asset calculation.

### First diagnostic action

When a return is implausibly large, inspect decimal, percentage, basis-point, currency, and per-share conventions before changing the model.

## Issue 9 — How is a simple return calculated?


A simple return compares the change in value with the beginning value. In words: divide the current price by the previous price and subtract one. Both prices must refer to the same asset, compatible field definition, currency, and adjacent holding interval.

The first observation has no earlier price inside the table, so its return remains missing. Filling that value with zero would create an unsupported holding interval.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [10]:
analysis_price = pd.Series(
    [100.0, 102.0, 101.0, 104.0, 103.0],
    index=pd.to_datetime(
        ["2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08", "2026-01-09"]
    ),
    name="Analysis_Price",
)

simple_return = analysis_price.pct_change(fill_method=None).rename("Simple_Return")
simple_table = pd.concat([analysis_price, simple_return], axis=1)
display(simple_table.round(6))

hand_return_second_date = 102.0 / 100.0 - 1
print("Hand calculation for second date:", hand_return_second_date)

assert pd.isna(simple_return.iloc[0])
assert np.isclose(simple_return.iloc[1], hand_return_second_date)
assert simple_return.notna().sum() == 4

,Analysis_Price,Simple_Return
2026-01-05,100.0,NaN
2026-01-06,102.0,0.020000
2026-01-07,101.0,-0.009804
2026-01-08,104.0,0.029703
2026-01-09,103.0,-0.009615


Hand calculation for second date: 0.020000000000000018


### Interpret the output


Five prices create four within-table holding intervals. The return labeled January 6 describes the interval from the January 5 price endpoint to the January 6 endpoint under this table convention.


### Discussion questions

1. Why does the first return remain missing rather than equal zero?
2. What start and end timestamps does the January 6 return represent?
3. How would the calculation change if one price used a different currency?
4. Why is the arithmetic correct but the result still invalid if the two fields use incompatible adjustments?

### Evidence to preserve

Save the price-return table, the hand substitution for one interval, the missing first return, and the interval labels.

### First diagnostic action

When a simple return is unexpected, display the two endpoint prices, their definitions, and their timestamps before checking later calculations.

## Issue 10 — How is a log return calculated?


A log return is the natural logarithm of the current price divided by the previous price. It is defined only when both price levels are positive. Log returns add across adjacent intervals, but they are not identical to simple returns.

The code calculates the log price difference and rejects nonpositive prices before applying the logarithm.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [11]:
if not analysis_price.gt(0).all():
    raise ValueError("Log returns require positive price levels.")

log_return = np.log(analysis_price).diff().rename("Log_Return")
log_table = pd.concat([analysis_price, simple_return, log_return], axis=1)
display(log_table.round(6))

hand_log_return = np.log(102.0 / 100.0)
print("Hand log return for second date:", hand_log_return)

assert pd.isna(log_return.iloc[0])
assert np.isclose(log_return.iloc[1], hand_log_return)
assert log_return.notna().sum() == 4

,Analysis_Price,Simple_Return,Log_Return
2026-01-05,100.0,NaN,NaN
2026-01-06,102.0,0.020000,0.019803
2026-01-07,101.0,-0.009804,-0.009852
2026-01-08,104.0,0.029703,0.029270
2026-01-09,103.0,-0.009615,-0.009662


Hand log return for second date: 0.01980262729617973


### Interpret the output


The second-date log return is slightly below the 2% simple return. For small changes they can look similar, but the difference becomes more visible for large price movements.


### Discussion questions

1. Why must price levels be positive before a log return is calculated?
2. What advantage does additivity give when adjacent intervals are combined?
3. Why should a report label log and simple returns rather than call both return?
4. Would a log return itself solve a corporate-action problem in the input price? Explain.

### Evidence to preserve

Save the positivity check, log-return table, hand calculation, and a sentence distinguishing the two return definitions.

### First diagnostic action

When a logarithm produces infinity or an error, inspect the original price for zero, negative, missing, or incorrectly scaled values.

## Issue 11 — How do simple and log returns reconcile?


Simple and log returns are linked exactly: one plus the simple return equals the exponential of the log return. This identity provides a useful implementation check. It does not decide which return definition best answers the financial question.

The example converts log returns back to simple returns and measures the largest numerical difference.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [12]:
simple_from_log = np.expm1(log_return).rename("Simple_From_Log")
reconciliation = pd.concat(
    [simple_return, log_return, simple_from_log],
    axis=1,
)
reconciliation["Absolute_Difference"] = (
    reconciliation["Simple_Return"] - reconciliation["Simple_From_Log"]
).abs()

display(reconciliation.round(12))
max_difference = reconciliation["Absolute_Difference"].max()
print("Maximum reconciliation difference:", max_difference)

assert max_difference < 1e-12

,Simple_Return,Log_Return,Simple_From_Log,Absolute_Difference
2026-01-05,NaN,NaN,NaN,NaN
2026-01-06,0.020000,0.019803,0.020000,0.0
2026-01-07,-0.009804,-0.009852,-0.009804,0.0
2026-01-08,0.029703,0.029270,0.029703,0.0
2026-01-09,-0.009615,-0.009662,-0.009615,0.0


Maximum reconciliation difference: 8.708311849403572e-16


### Interpret the output


The difference is at floating-point precision. If the two columns fail to reconcile materially, they were probably calculated from different prices, different intervals, or an incorrect percentage conversion.


### Discussion questions

1. What does a passed reconciliation check establish, and what does it not establish?
2. How could two correctly calculated return columns fail because their date labels refer to different intervals?
3. Why is a tiny floating-point difference acceptable but a one-percentage-point difference not?
4. Which return would you use for a wealth index, and how would you justify the choice?

### Evidence to preserve

Save the reconciliation table, tolerance, maximum difference, and one explanation for a possible failed identity.

### First diagnostic action

When the identity fails, compare the two source price columns and interval endpoints before changing numerical tolerances.

## Issue 12 — Why are multi-period simple returns compounded?


Simple returns describe proportional changes in wealth. When wealth changes across adjacent periods, the next return applies to the wealth remaining after the previous period. Multi-period simple return therefore multiplies one plus each return and subtracts one; adding simple returns ignores this changing base.

The difference between adding and compounding is small for short, low-volatility paths but can become material.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [13]:
period_returns = pd.Series(
    [0.10, -0.10, 0.05],
    index=["Period 1", "Period 2", "Period 3"],
    name="Simple_Return",
)

added_return = period_returns.sum()
compounded_return = (1.0 + period_returns).prod() - 1.0

display(period_returns.to_frame())
print("Added simple returns:", added_return)
print("Compounded simple return:", compounded_return)

manual_terminal_wealth = 1.0 * 1.10 * 0.90 * 1.05
assert np.isclose(compounded_return, manual_terminal_wealth - 1.0)
assert not np.isclose(added_return, compounded_return)

,Simple_Return
Period 1,0.10
Period 2,-0.10
Period 3,0.05


Added simple returns: 0.05
Compounded simple return: 0.03950000000000009


### Interpret the output


The added return is 5%, while the compounded return reflects the changing wealth base and is lower. A 10% gain followed by a 10% loss does not return wealth to its starting value.


### Discussion questions

1. Why do equal positive and negative percentage returns fail to cancel?
2. When might adding log returns be valid even though adding simple returns is not?
3. How would a missing middle period affect the interpretation of the compounded path?
4. What interval information is needed before three returns can be compounded together?

### Evidence to preserve

Save the return path, added result, compounded result, hand wealth calculation, and a sentence explaining the changing base.

### First diagnostic action

When multi-period results disagree, reconstruct wealth one interval at a time before inspecting summary formulas.

## Issue 13 — What does a wealth index show?


A wealth index shows how one unit of initial wealth changes along a return path under stated reinvestment and cash-flow assumptions. Starting at one makes proportional changes easy to inspect. The path reveals drawdowns and recovery patterns that a terminal return alone cannot show.

For a price-only series with no external cash flows and compatible endpoints, terminal wealth should equal final price divided by initial price.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [14]:
wealth_index = (1.0 + simple_return.fillna(0.0)).cumprod().rename("Wealth_Index")
wealth_table = pd.concat([analysis_price, simple_return, wealth_index], axis=1)
display(wealth_table.round(6))

terminal_from_returns = wealth_index.iloc[-1]
terminal_from_prices = analysis_price.iloc[-1] / analysis_price.iloc[0]

print("Terminal wealth from returns:", terminal_from_returns)
print("Endpoint price ratio:", terminal_from_prices)

assert np.isclose(wealth_index.iloc[0], 1.0)
assert np.isclose(terminal_from_returns, terminal_from_prices)

,Analysis_Price,Simple_Return,Wealth_Index
2026-01-05,100.0,NaN,1.00
2026-01-06,102.0,0.020000,1.02
2026-01-07,101.0,-0.009804,1.01
2026-01-08,104.0,0.029703,1.04
2026-01-09,103.0,-0.009615,1.03


Terminal wealth from returns: 1.03
Endpoint price ratio: 1.03


### Interpret the output


The first missing return is replaced with zero only for initializing the wealth index at one; it is not reclassified as an observed holding return. The endpoint identity would need a different interpretation if external cash flows or incompatible price adjustments were present.


### Discussion questions

1. Why is zero used for initialization but retained as missing in the return table?
2. What path information is lost when only terminal wealth is reported?
3. How would a deposit or withdrawal break the simple endpoint identity?
4. Why must a wealth chart state whether distributions and costs are included?

### Evidence to preserve

Save the wealth table, initialization rule, endpoint identity, and one interpretation of a rise and fall in the path.

### First diagnostic action

When terminal wealth does not equal the endpoint ratio, inspect return definition, cash flows, missing-value fills, and corporate-action treatment.

## Issue 14 — What is the difference between arithmetic and geometric mean returns?


The arithmetic mean averages periodic returns. The geometric mean is the constant per-period rate that reproduces the compounded terminal wealth over the same number of valid intervals. They answer different questions and need not be equal.

For simple returns greater than minus 100%, the geometric mean cannot exceed the arithmetic mean. It is strictly lower when the valid periodic returns are not all equal.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [15]:
valid_returns = simple_return.dropna()
arithmetic_mean = valid_returns.mean()
geometric_mean = (1.0 + valid_returns).prod() ** (1.0 / len(valid_returns)) - 1.0

reproduced_terminal_wealth = (1.0 + geometric_mean) ** len(valid_returns)
observed_terminal_wealth = (1.0 + valid_returns).prod()

summary_means = pd.Series(
    {
        "Arithmetic_mean": arithmetic_mean,
        "Geometric_mean": geometric_mean,
        "Observed_terminal_wealth": observed_terminal_wealth,
        "Wealth_from_geometric_mean": reproduced_terminal_wealth,
    }
)
display(summary_means)

assert geometric_mean <= arithmetic_mean + 1e-15
assert np.isclose(reproduced_terminal_wealth, observed_terminal_wealth)

Arithmetic_mean               0.007571
Geometric_mean                0.007417
Observed_terminal_wealth      1.030000
Wealth_from_geometric_mean    1.030000
dtype: float64

### Interpret the output


The geometric mean reproduces terminal wealth; the arithmetic mean describes the average one-period return. Neither should be annualized without stating the observation frequency and annualization convention.


### Discussion questions

1. Which mean answers the question of a constant per-period compounded rate?
2. Why does return variability create a gap between the two means?
3. What happens to the geometric mean if any simple return is minus 100%?
4. Why must the number and frequency of valid intervals accompany either mean?

### Evidence to preserve

Save both means, the reproduced-wealth identity, the valid interval count, and an explanation of the question answered by each statistic.

### First diagnostic action

When the geometric mean is undefined, inspect whether one plus any simple return is zero or negative and return to the source data.

## Issue 15 — Are dates parsed, unique, and sorted?


Financial calculations depend on chronological order. Text labels can sort incorrectly, duplicate timestamps can create ambiguous returns, and silent parsing assumptions can reverse day and month. Parsing should therefore be explicit, followed by uniqueness and monotonic-order checks.

The example contains unsorted input and a duplicate date. It audits the original order before creating a sorted view; it does not choose which duplicate is correct.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [16]:
date_records = pd.DataFrame(
    {
        "Date_text": ["2026-03-03", "2026-03-02", "2026-03-03", "2026-03-05"],
        "Price": [51.0, 50.0, 51.2, 52.0],
    }
)
date_records["Parsed_Date"] = pd.to_datetime(date_records["Date_text"], format="%Y-%m-%d")

date_audit = pd.Series(
    {
        "rows": len(date_records),
        "parse_failures": int(date_records["Parsed_Date"].isna().sum()),
        "duplicate_dates_beyond_first": int(date_records["Parsed_Date"].duplicated().sum()),
        "input_is_sorted": bool(date_records["Parsed_Date"].is_monotonic_increasing),
    }
)

display(date_records)
display(date_audit)
display(date_records.sort_values("Parsed_Date"))

assert date_audit["parse_failures"] == 0
assert date_audit["duplicate_dates_beyond_first"] == 1
assert not date_audit["input_is_sorted"]

,Date_text,Price,Parsed_Date
0,2026-03-03,51.0,2026-03-03
1,2026-03-02,50.0,2026-03-02
2,2026-03-03,51.2,2026-03-03
3,2026-03-05,52.0,2026-03-05


rows                                4
parse_failures                      0
duplicate_dates_beyond_first        1
input_is_sorted                 False
dtype: object

,Date_text,Price,Parsed_Date
1,2026-03-02,50.0,2026-03-02
0,2026-03-03,51.0,2026-03-03
2,2026-03-03,51.2,2026-03-03
3,2026-03-05,52.0,2026-03-05


### Interpret the output


Sorting can correct order but cannot resolve the duplicate. Selecting the first, last, average, or maximum would each create a different price history and requires a documented source rule.


### Discussion questions

1. Why should the original row order be preserved even after creating a sorted table?
2. What evidence would justify choosing one of the duplicate prices?
3. How could an ambiguous date format change the holding interval?
4. Why is a unique date index insufficient when multiple assets or venues share timestamps?

### Evidence to preserve

Save the original table, parsed dates, audit summary, sorted view, and an unresolved duplicate decision.

### First diagnostic action

When dates behave unexpectedly, display raw text, parsing format, parsed value, duplicate rows, and sorted order before calculating returns.

## Issue 16 — How should missing, stale, invalid, and extreme values be treated?


Missing, stale, invalid, and extreme describe different observations. A missing value is absent. A stale-price flag marks an unchanged value under a stated rule but does not prove an error. A nonpositive price requires investigation before an ordinary return calculation; a zero may represent total loss, a placeholder, or a data error, and a log return is not defined at zero. An extreme return is a review flag whose threshold is an analysis choice, not an automatic deletion rule.

The example preserves every original row and adds flags. The 25% extreme-return threshold is an artificial diagnostic setting.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [17]:
quality_prices = pd.Series(
    [100.0, 100.0, np.nan, 0.0, 150.0, 149.0, 210.0],
    index=pd.to_datetime(
        [
            "2026-04-01",
            "2026-04-02",
            "2026-04-03",
            "2026-04-06",
            "2026-04-07",
            "2026-04-08",
            "2026-04-09",
        ]
    ),
    name="Price",
)

quality_audit = quality_prices.to_frame()
quality_audit["missing"] = quality_audit["Price"].isna()
quality_audit["nonpositive"] = quality_audit["Price"].notna() & quality_audit["Price"].le(0)
quality_audit["unchanged_from_previous"] = quality_audit["Price"].eq(quality_audit["Price"].shift(1))
quality_audit["raw_simple_return"] = quality_audit["Price"].pct_change(fill_method=None)
quality_audit["infinite_return"] = np.isinf(quality_audit["raw_simple_return"])
quality_audit["finite_extreme_return"] = (
    np.isfinite(quality_audit["raw_simple_return"])
    & quality_audit["raw_simple_return"].abs().gt(0.25)
)

display(quality_audit)
print(
    quality_audit[
        [
            "missing",
            "nonpositive",
            "unchanged_from_previous",
            "infinite_return",
            "finite_extreme_return",
        ]
    ].sum()
)

assert quality_audit["missing"].sum() == 1
assert quality_audit["nonpositive"].sum() == 1
assert quality_audit["unchanged_from_previous"].sum() == 1
assert quality_audit["infinite_return"].sum() == 1
assert quality_audit["finite_extreme_return"].sum() == 1

,Price,missing,nonpositive,unchanged_from_previous,raw_simple_return,infinite_return,finite_extreme_return
2026-04-01,100.0,False,False,False,NaN,False,False
2026-04-02,100.0,False,False,True,0.000000,False,False
2026-04-03,NaN,True,False,False,NaN,False,False
2026-04-06,0.0,False,True,False,NaN,False,False
2026-04-07,150.0,False,False,False,inf,True,False
2026-04-08,149.0,False,False,False,-0.006667,False,False
2026-04-09,210.0,False,False,False,0.409396,False,True


missing                    1
nonpositive                1
unchanged_from_previous    1
infinite_return            1
finite_extreme_return      1
dtype: int64


### Interpret the output


The unchanged price is flagged but not declared wrong. The return from zero to 150 is infinite and cannot be interpreted until the zero is resolved. The finite increase from 149 to 210 separately triggers the illustrative 25% review threshold. Flags support investigation; they do not supply repairs.


### Discussion questions

1. When could an unchanged price be legitimate rather than stale?
2. Why should a missing value not be automatically forward-filled?
3. What financial event might explain an extreme return without making it erroneous?
4. Which decisions require provider evidence, and which are analysis conventions?

### Evidence to preserve

Save the original series, separate infinite and finite-extreme flags, the artificial threshold, unresolved cases, and the effect of any proposed treatment on retained rows.

### First diagnostic action

When an extreme or infinite return appears, inspect both endpoint prices and nearby corporate-action or trading-status records before filtering.

## Issue 17 — Is a missing calendar date a missing trading observation?


Markets do not necessarily trade every calendar day. Weekends, holidays, special closures, and asset-specific suspensions can create dates with no observation. A calendar gap becomes a data-quality problem only after comparison with the correct market calendar and session definition.

The example uses an explicitly artificial session schedule. It shows why a business-day generator is not authoritative evidence for any real exchange.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [18]:
calendar_days = pd.date_range("2026-01-02", "2026-01-08", freq="D")
observed_sessions = pd.DatetimeIndex(
    pd.to_datetime(["2026-01-02", "2026-01-05", "2026-01-06", "2026-01-08"])
)
artificial_closed_sessions = pd.DatetimeIndex(pd.to_datetime(["2026-01-07"]))

calendar_audit = pd.DataFrame(index=calendar_days)
calendar_audit["day_name"] = calendar_audit.index.day_name()
calendar_audit["observed"] = calendar_audit.index.isin(observed_sessions)
calendar_audit["weekend"] = calendar_audit.index.dayofweek >= 5
calendar_audit["artificial_scheduled_close"] = calendar_audit.index.isin(
    artificial_closed_sessions
)
calendar_audit["unresolved_gap"] = (
    ~calendar_audit["observed"]
    & ~calendar_audit["weekend"]
    & ~calendar_audit["artificial_scheduled_close"]
)

display(calendar_audit)
print("Unresolved gaps:", calendar_audit["unresolved_gap"].sum())

assert calendar_audit["unresolved_gap"].sum() == 0

,day_name,observed,weekend,artificial_scheduled_close,unresolved_gap
2026-01-02,Friday,True,False,False,False
2026-01-03,Saturday,False,True,False,False
2026-01-04,Sunday,False,True,False,False
2026-01-05,Monday,True,False,False,False
2026-01-06,Tuesday,True,False,False,False
2026-01-07,Wednesday,False,False,True,False
2026-01-08,Thursday,True,False,False,False


Unresolved gaps: 0


### Interpret the output


Weekend dates and the explicitly stated artificial closure are not treated as missing observations. For a real asset, the same conclusion would require the official calendar, time zone, and session information.


### Discussion questions

1. Why can a weekday gap be legitimate?
2. Which calendar should be used when assets trade on different venues?
3. How would an early close affect the meaning of an intraday observation?
4. Why is pandas business-day frequency not proof of an exchange session?

### Evidence to preserve

Save the observed dates, stated artificial closure, full calendar audit, and the source required for a real calendar.

### First diagnostic action

When a date appears missing, compare it with the authoritative venue calendar and asset trading status before filling or deleting rows.

## Issue 18 — Do timestamps and frequency labels describe the same interval?


A timestamp needs a time zone and boundary meaning. A label such as daily does not state whether it is a calendar-day close, exchange-session close, UTC aggregation, or provider-defined bar. Converting time zones changes labels but not the underlying instant; aggregating observations changes the interval represented.

The example converts artificial New York timestamps to UTC and separately constructs daily last observations from artificial intraday UTC prices. A daily last observation is not automatically an exchange-session close.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [19]:
local_times = pd.DatetimeIndex(
    pd.to_datetime(["2026-01-05 16:00", "2026-01-06 16:00"])
).tz_localize("America/New_York")
utc_times = local_times.tz_convert("UTC")

time_conversion = pd.DataFrame(
    {
        "Local_session_label": local_times.astype(str),
        "Same_instant_in_UTC": utc_times.astype(str),
    }
)
display(time_conversion)

intraday_prices = pd.Series(
    [100.0, 101.0, 102.0, 101.5],
    index=pd.DatetimeIndex(
        pd.to_datetime(
            [
                "2026-01-05 15:00+00:00",
                "2026-01-05 21:00+00:00",
                "2026-01-06 15:00+00:00",
                "2026-01-06 21:00+00:00",
            ],
            utc=True,
        )
    ),
    name="Artificial_Intraday_Price",
)
daily_last_utc = intraday_prices.resample("D").last().dropna().rename("Daily_Last_UTC")

display(intraday_prices.to_frame())
display(daily_last_utc.to_frame())

assert len(daily_last_utc) == 2
assert utc_times[0].tzinfo is not None

,Local_session_label,Same_instant_in_UTC
0,2026-01-05 16:00:00-05:00,2026-01-05 21:00:00+00:00
1,2026-01-06 16:00:00-05:00,2026-01-06 21:00:00+00:00


,Artificial_Intraday_Price
2026-01-05 15:00:00+00:00,100.0
2026-01-05 21:00:00+00:00,101.0
2026-01-06 15:00:00+00:00,102.0
2026-01-06 21:00:00+00:00,101.5


,Daily_Last_UTC
2026-01-05 00:00:00+00:00,101.0
2026-01-06 00:00:00+00:00,101.5


### Interpret the output


The local and UTC timestamps identify the same instants with different labels. The daily aggregation, however, applies a UTC calendar-day rule. A real exchange-session close may require a different boundary.


### Discussion questions

1. What information is lost when a time zone is removed?
2. Why can a UTC calendar day differ from a local exchange session?
3. Which aggregation functions would define Open, High, Low, Close, and Volume from intraday rows?
4. How could daylight-saving changes affect a fixed UTC-time assumption?

### Evidence to preserve

Save the original zone, converted timestamps, aggregation rule, resulting daily endpoints, and a statement of what daily means.

### First diagnostic action

When two daily datasets disagree, compare time zone, session boundary, included trading hours, and aggregation rule before comparing values.

## Issue 19 — Do multiple assets share a common holding interval?


A portfolio return combines asset returns that describe the same start and end boundaries. Calculating each asset return first and then joining by end date can hide different start dates when one asset has a missing price. A safer approach for this lesson is to build a common endpoint price table first and then calculate returns.

The artificial Asset E price is missing on March 4. The example does not forward-fill it.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [20]:
multi_asset_prices = pd.DataFrame(
    {
        "Asset_D": [80.0, 82.0, 81.0, 84.0, 83.0],
        "Asset_E": [40.0, 39.0, np.nan, 42.0, 43.0],
    },
    index=pd.to_datetime(
        ["2026-03-02", "2026-03-03", "2026-03-04", "2026-03-05", "2026-03-06"]
    ),
)

common_prices = multi_asset_prices.dropna(how="any")
common_returns = common_prices.pct_change(fill_method=None)

intervals = pd.DataFrame(
    {
        "Interval_start": common_prices.index.to_series().shift(1),
        "Interval_end": common_prices.index,
    },
    index=common_prices.index,
)

display(multi_asset_prices)
display(common_prices)
display(intervals)
display(common_returns.round(6))

assert len(common_prices) == 4
assert common_returns.notna().all(axis=1).sum() == 3

,Asset_D,Asset_E
2026-03-02,80.0,40.0
2026-03-03,82.0,39.0
2026-03-04,81.0,NaN
2026-03-05,84.0,42.0
2026-03-06,83.0,43.0


,Asset_D,Asset_E
2026-03-02,80.0,40.0
2026-03-03,82.0,39.0
2026-03-05,84.0,42.0
2026-03-06,83.0,43.0


,Interval_start,Interval_end
2026-03-02,NaT,2026-03-02
2026-03-03,2026-03-02,2026-03-03
2026-03-05,2026-03-03,2026-03-05
2026-03-06,2026-03-05,2026-03-06


,Asset_D,Asset_E
2026-03-02,NaN,NaN
2026-03-03,0.025000,-0.025000
2026-03-05,0.024390,0.076923
2026-03-06,-0.011905,0.023810


### Interpret the output


The return ending March 5 spans March 3 to March 5 for both assets. Asset D also has a March 3-to-March 4 daily return, but Asset E does not; combining those differently timed returns would not describe one portfolio interval.


### Discussion questions

1. Why is forward-filling Asset E not a neutral repair?
2. Which common interval is created after the March 4 row is excluded?
3. What information is lost by restricting all assets to common endpoints?
4. When might an alternative asynchronous-data method be justified, and what additional assumptions would it require?

### Evidence to preserve

Save the original table, missing-value audit, common endpoint table, explicit interval labels, and common return table.

### First diagnostic action

When component returns share an end label but seem inconsistent, display both start and end boundaries for every asset.

## Issue 20 — What does a one-period fixed-weight portfolio return mean?


A one-period fixed-weight portfolio return is the weighted sum of component returns over one common holding interval, using weights established before that interval begins. The weights must be finite, aligned to the asset labels, and sum to one under a fully invested no-leverage convention.

This calculation is portfolio arithmetic, not a method for choosing weights. Week 8 studies how weights are selected and rebalanced.


### Before running the example

Write one prediction about the output. Do not replace the prediction after seeing the result; add a revision underneath it.

In [21]:
weights = pd.Series({"Asset_D": 0.70, "Asset_E": 0.30}, name="Starting_Weight")

if not np.isfinite(weights).all():
    raise ValueError("Weights must be finite.")
if not np.isclose(weights.sum(), 1.0):
    raise ValueError("Weights must sum to one for this exercise.")
if set(weights.index) != set(common_returns.columns):
    raise ValueError("Weight labels must match the return columns.")

weights = weights.reindex(common_returns.columns)

portfolio_return = common_returns.mul(weights, axis=1).sum(
    axis=1,
    min_count=len(weights),
).rename("Portfolio_Return")

portfolio_table = common_returns.assign(Portfolio_Return=portfolio_return)
display(weights.to_frame())
display(portfolio_table.round(6))

march_5_hand = (
    0.70 * (84.0 / 82.0 - 1.0)
    + 0.30 * (42.0 / 39.0 - 1.0)
)
print("Hand portfolio return ending March 5:", march_5_hand)

assert pd.isna(portfolio_return.iloc[0])
assert np.isclose(portfolio_return.loc["2026-03-05"], march_5_hand)

,Starting_Weight
Asset_D,0.7
Asset_E,0.3


,Asset_D,Asset_E,Portfolio_Return
2026-03-02,NaN,NaN,NaN
2026-03-03,0.025000,-0.025000,0.01000
2026-03-05,0.024390,0.076923,0.04015
2026-03-06,-0.011905,0.023810,-0.00119


Hand portfolio return ending March 5: 0.0401500938086304


### Interpret the output


The first portfolio return remains missing because there is no preceding common endpoint. The March 5 calculation uses the same March 3-to-March 5 interval for both assets and the stated 70/30 beginning weights.


### Discussion questions

1. At what time must the weights be known for the March 3-to-March 5 return?
2. Why must weights be aligned by asset label rather than only by column position?
3. How would leverage or a cash holding change the weight constraints?
4. Why does this calculation not tell us whether 70/30 is a good weighting method?

### Evidence to preserve

Save the weight checks, component returns with interval labels, hand substitution, Python result, and a statement separating arithmetic from weight selection.

### First diagnostic action

When a portfolio return is missing or incorrect, inspect interval alignment, weight labels, weight sum, missing components, and the exact hand calculation.

# Independent practice issues

The four exercises below use new values and variable names. Preserve your prediction before coding, the complete input, calculations, checks, revisions, and unresolved questions. Starter cells intentionally do not contain solutions.

## Practice issue 1 — Select fields across corporate actions

Case A changes from price 120 with one share to price 40 with three shares after an artificial three-for-one split. Case B changes from price 75 to 73 and pays an artificial cash distribution of 4.

Calculate raw price return and position-value return for Case A. Calculate price return and simplified holding-period return for Case B. Then list the real provider fields required and one possible double-counting error.

Discussion: Which return answers each question? Which assumptions are artificial? What evidence would identify the event time? When would an adjusted field make a manual correction unnecessary?

In [22]:
practice_1_inputs = pd.DataFrame(
    {
        "Case": ["A_before", "A_after", "B_before", "B_after"],
        "Price": [120.0, 40.0, 75.0, 73.0],
        "Shares": [1.0, 3.0, 1.0, 1.0],
        "Cash_distribution": [0.0, 0.0, 0.0, 4.0],
    }
).set_index("Case")

display(practice_1_inputs)
print("Create new variables for the four required returns and add explicit assertions.")

,Price,Shares,Cash_distribution
Case,,,
A_before,120.0,1.0,0.0
A_after,40.0,3.0,0.0
B_before,75.0,1.0,0.0
B_after,73.0,1.0,4.0


Create new variables for the four required returns and add explicit assertions.


## Practice issue 2 — Reconcile returns and terminal wealth

Use the five artificial prices below. Calculate simple and log returns, reconcile their identity, construct a wealth index, verify the endpoint price ratio, and compare arithmetic with geometric mean return.

Discussion: How many holding intervals exist? Which first value remains missing? Which identity detects an interval error? What claim remains unsupported even if every check passes?

In [23]:
practice_2_price = pd.Series(
    [100.0, 104.0, 101.0, 106.0, 103.0],
    index=pd.to_datetime(
        ["2026-02-02", "2026-02-03", "2026-02-04", "2026-02-05", "2026-02-06"]
    ),
    name="Practice_2_Price",
)

display(practice_2_price.to_frame())
print("Predict terminal wealth and the ordering of the two means before calculating.")

,Practice_2_Price
2026-02-02,100.0
2026-02-03,104.0
2026-02-04,101.0
2026-02-05,106.0
2026-02-06,103.0


Predict terminal wealth and the ordering of the two means before calculating.


## Practice issue 3 — Audit a damaged financial table

Audit the table below without repairing it. Check parsing, order, duplicates, missing prices, nonpositive prices, and absent time-zone metadata. For each failed check, state why it matters, evidence needed before repair, proposed action or pending decision, and effect on the retained sample.

Discussion: Which issue prevents return calculation first? Which rows need provider evidence? Why is deleting all failed rows not a neutral solution? What should remain visible after a repair?

In [24]:
practice_3_records = pd.DataFrame(
    [
        ["2026-03-02 16:00", 50.0, 2_000],
        ["2026-03-03 16:00", 51.0, 2_100],
        ["2026-03-03 16:00", 51.2, 2_100],
        ["2026-03-05 16:00", np.nan, 1_900],
        ["2026-03-04 16:00", 0.0, 2_200],
    ],
    columns=["Timestamp", "Analysis_Price", "Volume"],
)

display(practice_3_records)
print("Preserve this original table and create a separate Boolean audit table.")

,Timestamp,Analysis_Price,Volume
0,2026-03-02 16:00,50.0,2000
1,2026-03-03 16:00,51.0,2100
2,2026-03-03 16:00,51.2,2100
3,2026-03-05 16:00,NaN,1900
4,2026-03-04 16:00,0.0,2200


Preserve this original table and create a separate Boolean audit table.


## Practice issue 4 — Align assets before calculating a portfolio return

Use the artificial prices below without forward-filling. Construct common endpoints, label every common holding interval, calculate returns, and calculate the return ending May 7 for beginning weights of 60% Asset F and 40% Asset G.

Discussion: Which dates are common endpoints? Which return intervals disappear? What is gained and lost by common-date restriction? Why must the first portfolio return remain missing?

In [25]:
practice_4_prices = pd.DataFrame(
    {
        "Asset_F": [90.0, 92.0, 91.0, 95.0, 94.0],
        "Asset_G": [45.0, np.nan, 46.0, 48.0, 47.0],
    },
    index=pd.to_datetime(
        ["2026-05-01", "2026-05-04", "2026-05-05", "2026-05-07", "2026-05-08"]
    ),
)
practice_4_weights = pd.Series({"Asset_F": 0.60, "Asset_G": 0.40})

display(practice_4_prices)
display(practice_4_weights.to_frame("Starting_Weight"))
print("Build the common price table before calculating either asset return.")

,Asset_F,Asset_G
2026-05-01,90.0,45.0
2026-05-04,92.0,NaN
2026-05-05,91.0,46.0
2026-05-07,95.0,48.0
2026-05-08,94.0,47.0


,Starting_Weight
Asset_F,0.6
Asset_G,0.4


Build the common price table before calculating either asset return.


# Completion evidence

The Week 2 notebook is complete when another student can:

- identify every required input, unit, timestamp meaning, and field definition;
- distinguish provider evidence from an access library and query specification;
- rerun every worked example without an execution error;
- reproduce return identities and hand calculations;
- explain every missing first return and every retained unresolved data problem;
- align multi-asset returns over common holding intervals;
- distinguish a fixed-weight arithmetic example from a method for selecting weights; and
- state why the notebook does not establish predictability, profitability, or future performance.

Submit the completed notebook or an exported report together with the Week 2 data-processing record. For the optional live yfinance example, submit the query record and diagnostic summary, not downloaded raw rows.